In [ ]:
import sys
import os
from pathlib import Path

if __name__ == "__main__":

    module_root = str(Path(os.getcwd()).parents[3])
    print(f"Module root: {module_root}")
    sys.path.insert(0, module_root)

    current_dir = str(Path(os.getcwd()).parents[2])
    print(f"root: {current_dir}")
    sys.path.insert(0, current_dir)

    if __package__ is None:
        __package__ = (
            "comfyui_image_scorer.external_modules.training_hyperparameters.notebooks"
        )


from ....shared.logger import configure_package_logging, get_logger, ModuleLogger

logger: ModuleLogger = get_logger(__name__)

#    fmt="[%(levelname)s] [%(funcName)s] %(asctime)s :\n%(message)s",


configure_package_logging(
    level=10,
    fmt="%(message)s [%(asctime)s]",
    datefmt="%H:%M:%S",  # Full date and time
    trim_level_len=1,  # Map INFO to INF, WARNING to WRN, etc.
    trim_module_len=5,  # Only show the final name of the module
    trim_func_len=10,  # Truncate function names exceeding 12 chars
    trim_msg_len=None,  # Truncate messages exceeding 40 chars
)


In [ ]:
from ..hyperparameter_optimizer import (
    load_training_data,
    reset_hyperparameters,
    hpo_cycle,
)
from ....shared.config import config

# Load options from the single source of truth: training_config.json
training_cfg = config["training"]

# Read-only values pulled from training config
NUM_STEPS = int(training_cfg["steps_per_cycle"])
NUM_CYCLES = int(training_cfg["cycles"])
MAX_COMBOS = int(training_cfg["max_combos"])
# ───────────────────────────────────────────────────────────────

# Load training data (no automatic retrain from config)
x, y = load_training_data(retrain=False, filter_comparisons=True)
print(f"Training data ready: X={x.shape}, y={y.shape}")


In [ ]:
# Guarded creation of base configs (top1..top4)
RESET = False  # set True to initialize base configs and persist to training_config.json
if RESET:
    # Force create base configs and persist to disk
    reset_hyperparameters()
    print("Base configs created: top1..top4 written to training_config.json")
else:
    print(
        "RESET is False: skipping creation of base configs. To create them, set RESET=True and re-run this cell."
    )
from IPython.display import display, HTML
import ipywidgets as widgets

display(HTML("""<style>
.widget-accordion .p-AccordionPanel-content,
.widget-accordion .lm-AccordionPanel-content,
.widget-accordion .jp-AccordionPanel-content {
  overflow-x: auto !important;
  background: transparent !important;
}
.widget-accordion .widget-output {
  background: transparent !important;
}
.widget-accordion .widget-output pre {
  background: transparent !important;
  white-space: pre-wrap !important;
  word-break: break-all !important;
}
</style>"""))

cycle_outputs = [widgets.Output() for _ in range(NUM_CYCLES)]
accordion = widgets.Accordion(children=cycle_outputs)

for i in range(NUM_CYCLES):
    accordion.set_title(i, f"HPO Cycle {i + 1}/{NUM_CYCLES}")

display(accordion)

for cycle in range(NUM_CYCLES):
    accordion.selected_index = cycle
    out = cycle_outputs[cycle]

    with out:
        print(f"=== Starting HPO Cycle {cycle + 1}/{NUM_CYCLES} ===")
        result = hpo_cycle(
            x, y, num_steps=NUM_STEPS, max_combos=MAX_COMBOS, cycle=cycle
        )
        print("HPO result summary:", result)
